# kashi — CTC bootstrap (Colab GPU)

Fine-tunes `rinna/japanese-hubert-base` with CTC on the karaoke label **sequences**
(timestamps unused → jitter-immune), then **forced-aligns** every song to produce
sharp per-frame labels for the local textless decoder (SIGNOFFS S9/S10).

**Before running:** `kashi colab pack` locally, upload `artifacts/colab_pack.zip`
to Drive as `MyDrive/Projects/krke_ctn/colab_pack.zip`. Runtime: **GPU** (T4 is fine, ~1-2 h).
**After:** download `MyDrive/Projects/krke_ctn/ctc_out.zip`, then locally:
`kashi colab integrate ~/Downloads/ctc_out.zip`.

In [ ]:
%%capture
!pip install -q "transformers>=4.44" soundfile
import torch, torchaudio, transformers
print(torch.__version__, transformers.__version__, torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Projects/krke_ctn/kashi'
DATA  = f'{DRIVE}/data'
import os, zipfile
os.makedirs(DATA, exist_ok=True)
if not os.path.exists(f'{DATA}/tokens.txt'):   # extract once; persists on Drive
    with zipfile.ZipFile(f'{DRIVE}/colab_pack.zip') as z:
        z.extractall(DATA)
print(sorted(os.listdir(DATA)))

In [ ]:
# ---- config ----
CHECKPOINT = 'rinna/japanese-hubert-base'
EPOCHS      = 12
LR_ENCODER  = 3e-5
LR_HEAD     = 1e-3
BATCH       = 4          # ~15 s crops; raise on A100
GRAD_ACCUM  = 2
CROP_S      = 15.0
SR          = 16000
FRAME_MS    = 20
DATA        = '/content/drive/MyDrive/Projects/krke_ctn/data'

In [ ]:
# ---- tokens & splits (from the pack; order matters: index 109 = <silence> = CTC blank) ----
TOKENS = open(f'{DATA}/tokens.txt').read().splitlines()
assert len(TOKENS) == 110 and TOKENS[-1] == '<silence>'
TOK2ID = {t: i for i, t in enumerate(TOKENS)}
BLANK = 109   # <silence> doubles as the CTC blank; sequences exclude silence rows
split = dict(l.split(':') for l in open(f'{DATA}/split.txt').read().splitlines())
TRAIN_IDS = [int(x) for x in split['train'].split(',')]
TEST_IDS  = [int(x) for x in split['test'].split(',')]
print(len(TRAIN_IDS), 'train /', len(TEST_IDS), 'test songs')

In [ ]:
# ---- crops: cut songs at silence rows into <=CROP_S chunks with token id sequences ----
import csv, soundfile as sf, numpy as np

def read_rows(song_id):
    with open(f'{DATA}/subtitles/{song_id}.csv', newline='') as f:
        return [dict(r) for r in csv.DictReader(f)]

def song_crops(song_id):
    rows = read_rows(song_id)
    crops, cur, t0 = [], [], None
    def flush(t_end):
        nonlocal cur, t0
        if cur and t_end - t0 >= 1.0:
            crops.append((t0, t_end, [TOK2ID[t] for t in cur]))
        cur, t0 = [], None
    for r in rows:
        tok = r['token']; s, e = float(r['start']), float(r['end'])
        excl = r.get('exclude', 'False').strip().lower() == 'true'
        if tok == '<silence>' or excl or tok not in TOK2ID or tok == '<silence>':
            if cur and (excl or (e - s) > 0.6):   # break at exclusions/long silences
                flush(s)
            continue
        if t0 is None: t0 = max(0.0, s - 0.1)
        if e - t0 > CROP_S:
            flush(s)
            t0 = max(0.0, s - 0.1)
        cur.append(tok)
        last_e = e
    if cur: flush(last_e + 0.1)
    return crops

CROPS = {sid: song_crops(sid) for sid in TRAIN_IDS + TEST_IDS}
print('crops:', sum(len(v) for v in CROPS.values()))

In [ ]:
# ---- dataset / loader ----
import random
from torch.utils.data import Dataset, DataLoader

class CropSet(Dataset):
    def __init__(self, ids):
        self.items = [(sid, c) for sid in ids for c in CROPS[sid]]
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        sid, (t0, t1, labels) = self.items[i]
        info = sf.info(f'{DATA}/vocals/{sid}.mp3')
        srr = info.samplerate
        wave, _ = sf.read(f'{DATA}/vocals/{sid}.mp3', start=int(t0*srr),
                          frames=int((t1-t0)*srr), dtype='float32', always_2d=True)
        wave = wave.mean(1)
        if srr != SR:
            wave = torchaudio.functional.resample(torch.from_numpy(wave), srr, SR).numpy()
        wave = (wave - wave.mean()) / (wave.std() + 1e-7)
        return torch.from_numpy(wave), torch.tensor(labels)

def collate(batch):
    waves, labels = zip(*batch)
    wl = torch.tensor([len(w) for w in waves])
    ll = torch.tensor([len(l) for l in labels])
    wpad = torch.zeros(len(waves), int(wl.max()))
    for i, w in enumerate(waves): wpad[i, :len(w)] = w
    return wpad, wl, torch.cat(labels), ll

train_dl = DataLoader(CropSet(TRAIN_IDS), batch_size=BATCH, shuffle=True,  collate_fn=collate, num_workers=2)
test_dl  = DataLoader(CropSet(TEST_IDS),  batch_size=BATCH, shuffle=False, collate_fn=collate, num_workers=2)

In [ ]:
# ---- model ----
from transformers import AutoModelForCTC
model = AutoModelForCTC.from_pretrained(
    CHECKPOINT, vocab_size=110, pad_token_id=BLANK, ctc_loss_reduction='mean',
    attn_implementation='sdpa').cuda()
model.freeze_feature_encoder()
head = [p for n, p in model.named_parameters() if 'lm_head' in n]
rest = [p for n, p in model.named_parameters() if 'lm_head' not in n and p.requires_grad]
opt = torch.optim.AdamW([{'params': rest, 'lr': LR_ENCODER}, {'params': head, 'lr': LR_HEAD}])
scaler = torch.amp.GradScaler()
ctc = torch.nn.CTCLoss(blank=BLANK, zero_infinity=True)

In [ ]:
# ---- SER (edit distance) for eval ----
def lev(a, b):
    if not a: return len(b)
    prev = list(range(len(b)+1))
    for i, x in enumerate(a, 1):
        cur = [i] + [0]*len(b)
        for j, y in enumerate(b, 1):
            cur[j] = min(prev[j]+1, cur[j-1]+1, prev[j-1] + (x != y))
        prev = cur
    return prev[-1]

@torch.no_grad()
def evaluate():
    model.eval(); d = n = 0
    for wpad, wl, labels, ll in test_dl:
        with torch.autocast('cuda', torch.float16):
            logits = model(wpad.cuda()).logits
        preds = logits.argmax(-1).cpu()
        off = 0
        for i in range(len(wl)):
            T_i = model._get_feat_extract_output_lengths(wl[i]).item()
            p = preds[i, :T_i]
            hyp = [int(x) for x in torch.unique_consecutive(p) if int(x) != BLANK]
            ref = labels[off:off+ll[i]].tolist(); off += ll[i]
            d += lev(hyp, ref); n += len(ref)
    model.train(); return d / max(1, n)

In [ ]:
# ---- train ----
best = 9e9
for epoch in range(1, EPOCHS+1):
    tot = steps = 0
    for k, (wpad, wl, labels, ll) in enumerate(train_dl):
        with torch.autocast('cuda', torch.float16):
            logits = model(wpad.cuda()).logits          # [B, T, 110]
        logp = torch.log_softmax(logits.float(), -1).transpose(0, 1)
        out_len = model._get_feat_extract_output_lengths(wl).cuda()
        loss = ctc(logp, labels.cuda(), out_len, ll.cuda()) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (k+1) % GRAD_ACCUM == 0:
            scaler.step(opt); scaler.update(); opt.zero_grad()
        tot += float(loss) * GRAD_ACCUM; steps += 1
    ser = evaluate()
    print(f'epoch {epoch}/{EPOCHS} loss={tot/steps:.4f} test_SER={ser:.4f}', flush=True)
    if ser < best:
        best = ser
        model.save_pretrained('/content/ctc_best')
print('best test SER (greedy, no LM):', best)

In [ ]:
# ---- forced alignment of EVERY labeled song (S9: training bootstrap only) ----
import numpy as np, os
model.load_state_dict(AutoModelForCTC.from_pretrained('/content/ctc_best').state_dict())
model.eval()
os.makedirs('/content/out/fa_labels', exist_ok=True)
os.makedirs('/content/out/fa_segments', exist_ok=True)

@torch.no_grad()
def song_emissions(sid, chunk_s=30.0, ov_s=2.0):
    info = sf.info(f'{DATA}/vocals/{sid}.mp3'); srr = info.samplerate
    wave, _ = sf.read(f'{DATA}/vocals/{sid}.mp3', dtype='float32', always_2d=True)
    wave = wave.mean(1)
    if srr != SR:
        wave = torchaudio.functional.resample(torch.from_numpy(wave), srr, SR).numpy()
    wave = (wave - wave.mean()) / (wave.std() + 1e-7)
    hop = SR * FRAME_MS // 1000
    T = len(wave) // hop
    parts, step = [], int((chunk_s - ov_s) * SR)
    for s in range(0, len(wave), step):
        piece = torch.from_numpy(wave[max(0, s-int(ov_s*SR)) : s+int(chunk_s*SR)]).float()
        with torch.autocast('cuda', torch.float16):
            lg = model(piece[None].cuda()).logits[0].float().cpu()
        lead = (s - max(0, s-int(ov_s*SR))) // hop
        keep = min(int((chunk_s-ov_s)*SR)//hop, len(lg)-lead)
        parts.append(lg[lead:lead+keep])
        if s + int(chunk_s*SR) >= len(wave): break
    em = torch.cat(parts)[:T]
    return torch.log_softmax(em, -1), T

for sid in TRAIN_IDS + TEST_IDS:
    rows = read_rows(sid)
    seq = [TOK2ID[r['token']] for r in rows
           if r['token'] in TOK2ID and r['token'] != '<silence>'
           and r.get('exclude','False').strip().lower() != 'true']
    if not seq: continue
    em, T = song_emissions(sid)
    try:
        path, scores = torchaudio.functional.forced_align(
            em[None], torch.tensor([seq]), blank=BLANK)
    except Exception as e:
        print(sid, 'FA failed:', e); continue
    frame = path[0].numpy().astype(np.int16)      # class per frame; blank=109=<silence>
    np.save(f'/content/out/fa_labels/{sid}.npy', frame)
    # token segments csv
    with open(f'/content/out/fa_segments/{sid}.csv', 'w') as f:
        f.write('start,end,token,conf\n')
        t = 0
        sc = scores[0].exp().numpy()
        while t < T:
            u = t
            while u < T and frame[u] == frame[t]: u += 1
            if frame[t] != BLANK:
                f.write(f'{t*0.02:.3f},{u*0.02:.3f},{TOKENS[frame[t]]},{sc[t:u].mean():.3f}\n')
            t = u
    print(sid, 'ok', flush=True)

In [ ]:
# ---- package results to Drive ----
import json, shutil
shutil.copytree('/content/ctc_best', '/content/out/ctc_model', dirs_exist_ok=True)
json.dump({'checkpoint': CHECKPOINT, 'best_test_ser_greedy': best, 'epochs': EPOCHS},
          open('/content/out/metrics.json', 'w'))
shutil.make_archive(f'{DRIVE}/ctc_out', 'zip', '/content/out')
print('wrote', f'{DRIVE}/ctc_out.zip — download and run: kashi colab integrate ctc_out.zip')